<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-04-image-and-multimodal-generation/lesson-4.6-video-and-3d-generation/notebooks/exercises-4_6.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 4.6: Video & 3D Generation

*Module 4 · Image, Vision & Multimodal AI*

> AI can now generate photorealistic videos from text and reconstruct full 3D scenes from phone photos. This lesson explains how it works, who's building it, and how to use it today.

---

## About this notebook

This notebook contains the **7 runnable code examples** from the published lesson `lesson-4.6.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — How Video Generation Works — Images + Time — `part1_video_diffusion_concept.py`
2. Step 3 — API Demos — Generate Videos Programmatically — `part3a_runway_api.py`
3. Step 3 — API Demos — Generate Videos Programmatically — `part3b_image_to_video.py`
4. Step 3 — API Demos — Generate Videos Programmatically — `part3c_svd_local.py`
5. Step 4 — 3D Gaussian Splatting — Photos Become Explorable 3D Worlds — `part4a_gaussian_concept.py`
6. Step 4 — 3D Gaussian Splatting — Photos Become Explorable 3D Worlds — `part4b_run_gaussian_splatting.py`
7. Step 5 — Production Workflows — When and How to Use Video AI — `part5_video_pipeline.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q torch numpy matplotlib requests pillow


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export RUNWAY_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("RUNWAY_API_KEY", "YOUR_RUNWAY_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · How Video Generation Works — Images + Time

A video is just 24 images per second with one critical constraint: they must be consistent.

**`part1_video_diffusion_concept.py`** · _Block 1 of 7_

VIDEO DIFFUSION: The core concept in code — A video is a tensor: (batch, frames, channels, height, width)


In [ ]:
# =============================================
# VIDEO DIFFUSION: The core concept in code
# A video is a tensor: (batch, frames, channels, height, width)
# We denoise across BOTH spatial and temporal dimensions.
# =============================================

import torch

# An image is: (batch, channels, height, width)
image = torch.randn(1, 3, 512, 512)
print(f"Image shape:  {image.shape}")

# A video is: (batch, FRAMES, channels, height, width)
video = torch.randn(1, 24, 3, 512, 512)  # 24 frames = 1 second at 24fps
print(f"Video shape:  {video.shape}")
print(f"Video size:   {video.numel() * 4 / 1e9:.1f} GB (float32)")

# ── This is why video diffusion is so expensive ──
# A 512×512 image = 0.79M pixels
# A 1-second video (24 frames) = 18.9M pixels (24x more!)
# A 10-second video = 189M pixels (240x more than one image!)

print(f"\nPixels in 1 image:       {512*512*3:>12,}")
print(f"Pixels in 1s video:      {24*512*512*3:>12,}  (24x more)")
print(f"Pixels in 10s video:     {240*512*512*3:>12,}  (240x more)")

# ── Sora's approach: spacetime patches ──
# Instead of denoising raw pixels, compress into patches
# A "spacetime patch" covers e.g., 4 frames × 16×16 pixels
patch_size_spatial = 16
patch_size_temporal = 4
num_patches = (24 // patch_size_temporal) * (512 // patch_size_spatial) ** 2
print(f"\nSpacetime patches: {num_patches:,} (much more manageable!)")

print("""
Key insight: Video generation is computationally brutal.
  - 10 seconds of 1080p video = billions of values to denoise
  - That's why Sora needs massive GPU clusters
  - And why video generation costs $0.20-$2.00 per clip
  - Latent compression + patches make it possible (but still expensive)
""")


> **What just happened?** We calculated why video generation is so hard: a 10-second video has 240× more data than a single image. That's why Sora uses spacetime patches (compressing 4 frames × 16×16 pixels into one token) and latent space (compressing each frame 8×). Even with these tricks, generating one 10-second clip requires hundreds of GPU-seconds. This is the fundamental constraint driving the entire field.


### Step 3 · API Demos — Generate Videos Programmatically

Text in, video out. Let's use real APIs to generate video clips.

**`part3a_runway_api.py`** · _Block 2 of 7_

RUNWAY GEN-3: Generate videos via API — pip install runwayml


In [ ]:
# =============================================
# RUNWAY GEN-3: Generate videos via API
# pip install runwayml
# Get API key from: https://dev.runwayml.com
# =============================================

from runwayml import RunwayML
import time, os

client = RunwayML(api_key=os.getenv("RUNWAY_API_KEY"))

# ── Text-to-Video ──
print("Generating video from text prompt...\n")

task = client.image_to_video.create(
    model="gen3a_turbo",
    prompt_text="""A majestic Bengal tiger walking slowly through a misty 
Indian forest at sunrise. Golden light filters through the trees. 
Cinematic, slow motion, 4K quality. Camera slowly dollies forward.""",
    duration=5,  # seconds (5 or 10)
)

# Poll until complete
print(f"Task ID: {task.id}")
while True:
    status = client.tasks.retrieve(task.id)
    print(f"  Status: {status.status}")
    if status.status == "SUCCEEDED":
        print(f"  Video URL: {status.output[0]}")
        break
    elif status.status == "FAILED":
        print(f"  Error: {status.failure}")
        break
    time.sleep(5)

print("""
Runway Gen-3 pricing (approximate):
  gen3a_turbo:  ~$0.25 per 5-second clip
  gen3a:        ~$0.50 per 5-second clip (higher quality)
  
Duration options: 5 seconds or 10 seconds
Resolution: up to 1280×768
""")


**`part3b_image_to_video.py`** · _Block 3 of 7_

IMAGE-TO-VIDEO: Animate a still image — Give it a photo + describe the motion you want.


In [ ]:
# =============================================
# IMAGE-TO-VIDEO: Animate a still image
# Give it a photo + describe the motion you want.
# =============================================

import base64

# Encode your image as base64
with open("product_photo.jpg", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

# Generate video from image + motion prompt
task = client.image_to_video.create(
    model="gen3a_turbo",
    prompt_image=f"data:image/jpeg;base64,{image_b64}",
    prompt_text="Camera slowly orbits around the product. Soft studio lighting. Smooth motion.",
    duration=5,
)

print(f"Animating image → video: Task {task.id}")

print("""
Image-to-video use cases:
  • Product demos: still product photo → 360° orbit video
  • Social media: static post → eye-catching animated post
  • Real estate: interior photo → smooth walkthrough
  • Fashion: model photo → runway walk animation
  • E-commerce: product grid → animated showcase
""")


**`part3c_svd_local.py`** · _Block 4 of 7_

STABLE VIDEO DIFFUSION: Run locally, free! — Image → short video clip on your own GPU.


In [ ]:
# =============================================
# STABLE VIDEO DIFFUSION: Run locally, free!
# Image → short video clip on your own GPU.
# Needs ~8 GB VRAM minimum.
# =============================================

from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import export_to_video
from PIL import Image
import torch

# Load model
pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=torch.float16,
).to("cuda")

# Load your starting image
image = Image.open("landscape.jpg").resize((1024, 576))

# Generate video frames
print("Generating video frames (this takes 2-5 minutes on a T4)...")
frames = pipe(
    image,
    num_frames=25,           # number of frames to generate
    decode_chunk_size=8,     # lower = less VRAM, slower
    motion_bucket_id=127,    # amount of motion (0=still, 255=wild)
    noise_aug_strength=0.02, # how much to augment the starting image
    num_inference_steps=25,  # denoising steps per frame
).frames[0]

# Export to video file
export_to_video(frames, "svd_output.mp4", fps=7)
print("Video saved: svd_output.mp4")

print("""
SVD controls:
  motion_bucket_id (0-255): controls how much things move
    0-50:   subtle motion (clouds drifting, water rippling)
    50-150: moderate motion (camera pan, person walking)
    150-255: dramatic motion (action scenes, fast movement)
    
  noise_aug_strength (0-1): how much to deviate from input image
    0.0:  stay very close to original
    0.02: slight animation (recommended)
    0.1+: more creative but less faithful
""")


> **What just happened?** Three ways to generate video: (1) Runway API — text-to-video, 5 seconds, ~$0.25 per clip, highest quality. (2) Runway image-to-video — animate a still photo with described motion, great for product demos. (3) Stable Video Diffusion — free, runs locally on your GPU, image-to-video with controllable motion intensity. Each serves a different need: Runway for quality, SVD for privacy and cost.


### Step 4 · 3D Gaussian Splatting — Photos Become Explorable 3D Worlds

Take 30 photos of an object or scene → get a 3D model you can fly through from any angle.

**`part4a_gaussian_concept.py`** · _Block 5 of 7_

GAUSSIAN SPLATTING: Core concept in code — What IS a 3D Gaussian? Let's build one.


In [ ]:
# =============================================
# GAUSSIAN SPLATTING: Core concept in code
# What IS a 3D Gaussian? Let's build one.
# =============================================

import torch
import numpy as np
import matplotlib.pyplot as plt

# A single 3D Gaussian is defined by:
class Gaussian3D:
    def __init__(self, position, color, opacity, scale):
        self.position = torch.tensor(position, dtype=torch.float32)   # [x, y, z]
        self.color = torch.tensor(color, dtype=torch.float32)         # [r, g, b]
        self.opacity = torch.tensor(opacity, dtype=torch.float32)     # 0-1
        self.scale = torch.tensor(scale, dtype=torch.float32)         # [sx, sy, sz]
    
    def __repr__(self):
        return f"Gaussian(pos={self.position.tolist()}, color={self.color.tolist()}, opacity={self.opacity:.2f})"

# A scene = millions of these Gaussians
scene = [
    Gaussian3D(position=[0.0, 0.0, 0.0], color=[1.0, 0.0, 0.0], opacity=0.9, scale=[0.5, 0.5, 0.5]),
    Gaussian3D(position=[1.0, 0.5, 0.0], color=[0.0, 1.0, 0.0], opacity=0.8, scale=[0.3, 0.3, 0.3]),
    Gaussian3D(position=[-0.5, 1.0, 0.5], color=[0.0, 0.0, 1.0], opacity=0.7, scale=[0.4, 0.6, 0.2]),
]

print(f"Scene: {len(scene)} Gaussians (real scenes have millions!)\n")
for g in scene:
    print(f"  {g}")

print("""
A real Gaussian Splatting scene:
  - Indoor room:  ~500K-2M Gaussians
  - Outdoor scene: ~2M-5M Gaussians
  - Full building:  ~5M-10M Gaussians
  
Each Gaussian has ~60 parameters (position, rotation, scale, 
color spherical harmonics, opacity). A 2M Gaussian scene = 
~120M parameters = ~500 MB of data.

But it renders at 100+ FPS — faster than traditional 3D!
""")


**`part4b_run_gaussian_splatting.py`** · _Block 6 of 7_

RUN GAUSSIAN SPLATTING on your own photos — Two options: the original implementation or


In [ ]:
# =============================================
# RUN GAUSSIAN SPLATTING on your own photos
# Two options: the original implementation or
# a user-friendly tool like Luma AI or Polycam.
# =============================================

# ── OPTION 1: Luma AI API (easiest, cloud-based) ──
import requests

# Upload photos to Luma AI for 3D reconstruction
# API: https://docs.lumalabs.ai/

luma_api_key = "your_luma_api_key"

# Create a new capture from uploaded photos
# response = requests.post(
#     "https://webapp.engineeringlumalabs.com/api/v3/captures",
#     headers={"Authorization": f"luma-api-key={luma_api_key}"},
#     json={"title": "My Room", "type": "gaussian_splatting"}
# )

print("""
Option 1: Luma AI (cloud, easiest)
  1. Download Luma AI app on your phone
  2. Walk around an object/room recording video
  3. Upload → wait 10-30 minutes
  4. Get an interactive 3D scene you can share via URL
  Free tier available, paid for higher quality.

Option 2: Original code (local, full control)
  git clone https://github.com/graphdeco-inria/gaussian-splatting
  cd gaussian-splatting
  pip install -r requirements.txt
  
  # Step 1: Extract camera positions from photos
  python convert.py -s ./my_photos
  
  # Step 2: Train Gaussian Splatting (30 min - 2 hours on GPU)
  python train.py -s ./my_photos --iterations 30000
  
  # Step 3: View the result
  python viewer.py -m ./output
  
  Needs: NVIDIA GPU with 8+ GB VRAM, COLMAP for SfM.

Option 3: Nerfstudio (local, user-friendly)
  pip install nerfstudio
  ns-process-data images --data ./my_photos
  ns-train splatfacto --data ./my_photos
  ns-viewer --load-config outputs/.../config.yml
  
  More user-friendly than the original, good documentation.
""")

# ── OPTION 4: Polycam API (mobile-first) ──
print("""
Option 4: Polycam (mobile app, great for quick scans)
  1. Download Polycam on iPhone (LiDAR) or Android
  2. Scan an object or room
  3. Export as Gaussian Splat, mesh, or point cloud
  4. View in web browser or import into game engine
  
Best for: quick product scans, room scans, real estate
""")


> **What just happened?** We explored the data structure of Gaussian Splatting (each Gaussian = position + color + opacity + scale = ~60 parameters, scenes have millions of them) and saw 4 practical ways to create 3D scenes: Luma AI (easiest, phone app), original research code (full control), Nerfstudio (user-friendly local), and Polycam (mobile-first). The key insight: you can turn 30 phone photos into an explorable 3D world that renders at 100+ FPS.


### Step 5 · Production Workflows — When and How to Use Video AI

Practical decision framework: which tool, what cost, what quality to expect.

**`part5_video_pipeline.py`** · _Block 7 of 7_

PRODUCTION VIDEO PIPELINE — Batch-generate product videos from images.


In [ ]:
# =============================================
# PRODUCTION VIDEO PIPELINE
# Batch-generate product videos from images.
# =============================================

import os, time, json
from pathlib import Path
from runwayml import RunwayML

class VideoProductionPipeline:
    """Batch video generation for e-commerce products."""
    
    def __init__(self):
        self.client = RunwayML(api_key=os.getenv("RUNWAY_API_KEY"))
        self.results = []
    
    def generate_product_video(self, image_path: str, product_name: str,
                               motion: str = "orbit") -> dict:
        """Generate a product showcase video from a photo."""
        
        motion_prompts = {
            "orbit": "Camera slowly orbits 360 degrees around the product. Studio lighting. Smooth.",
            "zoom": "Camera slowly zooms in revealing intricate details. Soft focus background.",
            "lifestyle": "Product in use in a natural setting. Warm lighting. Lifestyle commercial feel.",
            "dramatic": "Dramatic reveal with spotlight. Dark background. Cinematic. Slow motion.",
        }
        
        import base64
        with open(image_path, "rb") as f:
            img_b64 = base64.b64encode(f.read()).decode()
        
        task = self.client.image_to_video.create(
            model="gen3a_turbo",
            prompt_image=f"data:image/jpeg;base64,{img_b64}",
            prompt_text=motion_prompts.get(motion, motion_prompts["orbit"]),
            duration=5,
        )
        
        return {"task_id": task.id, "product": product_name, "motion": motion}
    
    def batch_generate(self, products: list[dict]):
        """Generate videos for multiple products."""
        print(f"Generating {len(products)} product videos...\n")
        
        tasks = []
        for p in products:
            result = self.generate_product_video(p["image"], p["name"], p.get("motion", "orbit"))
            tasks.append(result)
            print(f"  Submitted: {p['name']} ({result['motion']})")
        
        # Wait for all to complete
        print("\nWaiting for generation...")
        for t in tasks:
            while True:
                status = self.client.tasks.retrieve(t["task_id"])
                if status.status in ["SUCCEEDED", "FAILED"]:
                    t["status"] = status.status
                    t["url"] = status.output[0] if status.status == "SUCCEEDED" else None
                    emoji = "✅" if status.status == "SUCCEEDED" else "❌"
                    print(f"  {emoji} {t['product']}: {status.status}")
                    break
                time.sleep(5)
        
        return tasks

# ── Run it ──
pipeline = VideoProductionPipeline()

products = [
    {"name": "Running Shoes", "image": "shoes.jpg", "motion": "orbit"},
    {"name": "Headphones", "image": "headphones.jpg", "motion": "dramatic"},
    {"name": "Backpack", "image": "backpack.jpg", "motion": "lifestyle"},
]

# results = pipeline.batch_generate(products)

print("""
Cost estimate for 100 products:
  100 products × 4 motion types × $0.25 = $100
  That's $100 for 400 professional product videos.
  A human videographer would charge $10,000+ for the same work.
""")


> **What just happened?** We built a production pipeline that takes product photos and batch-generates showcase videos with 4 motion types: orbit (360° rotation), zoom (detail reveal), lifestyle (in-use context), and dramatic (cinematic spotlight). At $0.25 per clip, 400 professional product videos cost $100. A human videographer would charge $10,000+. This is how e-commerce companies are automating product video creation.


---

## Wrap-up

You've walked through all 7 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
